# Tahap 3: Case Retrieval
**Domain:** Pidana Umum - Pemalsuan (Pasal 263 & 266 KUHP)

Notebook ini mencakup:
1. Load dataset dari tahap 2
2. Train/test split
3. TF-IDF + SVM
4. IndoBERT + cosine similarity dengan **chunking**
5. Fungsi `retrieve()`
6. Buat ground-truth queries.json
7. Pengujian awal
8. Simpan semua model

## Setup

In [1]:
import os

BASE_DIR = r'C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan'

PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
EVAL_DIR      = os.path.join(BASE_DIR, 'data', 'eval')
MODEL_DIR     = os.path.join(BASE_DIR, 'models')

for d in [EVAL_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print('Path siap.')

Path siap.


## 1. Install & Import

In [2]:
%pip install transformers torch scikit-learn -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import torch
from transformers import AutoTokenizer, AutoModel
from typing import List, Tuple

print('Import selesai.')
print(f'GPU tersedia: {torch.cuda.is_available()}')

Import selesai.
GPU tersedia: False


## 2. Load Dataset

In [4]:
CSV_PATH = os.path.join(PROCESSED_DIR, 'cases.csv')
df = pd.read_csv(CSV_PATH)

df = df[df['pasal_terbukti'].notna()].reset_index(drop=True)

def simplify_label(pasal):
    if pasal == 'bebas':
        return 'bebas'
    if '266' in str(pasal):
        return 'pasal 266'
    if 'ayat (2)' in str(pasal):
        return 'pasal 263 ayat (2)'
    return 'pasal 263 ayat (1)'

df['label_svm'] = df['pasal_terbukti'].apply(simplify_label)

print(f'Total dokumen: {len(df)}')
print(f'Distribusi label_svm:')
print(df['label_svm'].value_counts())

Total dokumen: 36
Distribusi label_svm:
label_svm
pasal 263 ayat (1)    19
pasal 266              6
bebas                  6
pasal 263 ayat (2)     5
Name: count, dtype: int64


## 3. Train/Test Split

In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label_svm']
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)} dokumen')
print(train_df['pasal_terbukti'].value_counts())
print()
print(f'Test: {len(test_df)} dokumen')
print(test_df['pasal_terbukti'].value_counts())

train_df[['case_id', 'no_perkara', 'pasal_terbukti']].to_csv(
    os.path.join(EVAL_DIR, 'train_ids.csv'), index=False)
test_df[['case_id', 'no_perkara', 'pasal_terbukti']].to_csv(
    os.path.join(EVAL_DIR, 'test_ids.csv'), index=False)
print('Split disimpan.')

Train: 28 dokumen
pasal_terbukti
pasal 263 ayat (1)    15
bebas                  4
pasal 263 ayat (2)     4
pasal 266 ayat (1)     4
pasal 266 ayat (2)     1
Name: count, dtype: int64

Test: 8 dokumen
pasal_terbukti
pasal 263 ayat (1)    4
bebas                 2
pasal 266 ayat (1)    1
pasal 263 ayat (2)    1
Name: count, dtype: int64
Split disimpan.


## 4. Pendekatan 1: TF-IDF + SVM


In [6]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        strip_accents='unicode'
    )),
    ('svm', SVC(
        kernel='linear',
        C=1.0,
        probability=True,
        random_state=42
    ))
])

X_train = train_df['text_full'].tolist()
y_train = train_df['label_svm'].tolist()
X_test  = test_df['text_full'].tolist()
y_test  = test_df['label_svm'].tolist()

print('Pipeline TF-IDF + SVM siap.')

Pipeline TF-IDF + SVM siap.


In [7]:
print('=== 5-Fold Cross Validation (Train Set) ===')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_preds = cross_val_predict(svm_pipeline, X_train, y_train, cv=cv)

labels = sorted(train_df['pasal_terbukti'].unique())
print(classification_report(y_train, cv_preds, labels=labels))

=== 5-Fold Cross Validation (Train Set) ===


c:\Users\CENTRALGALAXY\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


                    precision    recall  f1-score   support

             bebas       1.00      0.50      0.67         4
pasal 263 ayat (1)       0.62      1.00      0.77        15
pasal 263 ayat (2)       0.00      0.00      0.00         4
pasal 266 ayat (1)       0.00      0.00      0.00         0
pasal 266 ayat (2)       0.00      0.00      0.00         0

         micro avg       0.65      0.74      0.69        23
         macro avg       0.33      0.30      0.29        23
      weighted avg       0.58      0.74      0.62        23



c:\Users\CENTRALGALAXY\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\CENTRALGALAXY\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\CENTRALGALAXY\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.


In [8]:
svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)

labels = sorted(test_df['pasal_terbukti'].unique())
print('=== Evaluasi SVM pada Test Set ===')
print(classification_report(y_test, y_pred_svm, labels=labels))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_svm, labels=labels))
print('Labels:', labels)

=== Evaluasi SVM pada Test Set ===
                    precision    recall  f1-score   support

             bebas       1.00      1.00      1.00         2
pasal 263 ayat (1)       0.80      1.00      0.89         4
pasal 263 ayat (2)       0.00      0.00      0.00         1
pasal 266 ayat (1)       0.00      0.00      0.00         0

         micro avg       0.86      0.86      0.86         7
         macro avg       0.45      0.50      0.47         7
      weighted avg       0.74      0.86      0.79         7

Confusion Matrix:
[[2 0 0 0]
 [0 4 0 0]
 [0 1 0 0]
 [0 0 0 0]]
Labels: ['bebas', 'pasal 263 ayat (1)', 'pasal 263 ayat (2)', 'pasal 266 ayat (1)']


c:\Users\CENTRALGALAXY\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\CENTRALGALAXY\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\CENTRALGALAXY\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.


In [9]:
tfidf_retriever = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents='unicode'
)
train_tfidf = tfidf_retriever.fit_transform(X_train)
print(f'TF-IDF retriever matrix shape: {train_tfidf.shape}')

TF-IDF retriever matrix shape: (28, 5000)


In [10]:
def retrieve_tfidf(query: str, k: int = 5) -> List[Tuple[str, float]]:
    query_clean  = query.lower().strip()
    query_vec    = tfidf_retriever.transform([query_clean])
    similarities = cosine_similarity(query_vec, train_tfidf).flatten()
    top_k_idx    = similarities.argsort()[::-1][:k]
    return [
        (train_df.iloc[idx]['case_id'], float(similarities[idx]))
        for idx in top_k_idx
    ]

print('Test TF-IDF retrieve:')
for cid, score in retrieve_tfidf(X_test[0], k=5):
    print(f'  {cid}: {score:.4f}')

Test TF-IDF retrieve:
  case_014: 0.4861
  case_029: 0.4716
  case_030: 0.4627
  case_011: 0.4566
  case_019: 0.4552


## 5. Pendekatan 2: IndoBERT + Cosine Similarity


In [11]:
MODEL_NAME = 'indobenchmark/indobert-base-p1'
print(f'Loading {MODEL_NAME}...')

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)
bert_model.eval()

print(f'Model loaded. Device: {device}')

Loading indobenchmark/indobert-base-p1...


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded. Device: cpu


In [12]:
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50


def get_bert_embedding_single(text: str) -> np.ndarray:
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=512,
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

    attention_mask       = inputs['attention_mask']
    token_embeddings     = outputs.last_hidden_state
    input_mask_expanded  = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()).float()
    embedding = torch.sum(token_embeddings * input_mask_expanded, 1)
    embedding = embedding / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    return embedding.cpu().numpy()[0]


def get_bert_embedding(text: str,
                       chunk_size: int = CHUNK_SIZE,
                       overlap: int = CHUNK_OVERLAP) -> np.ndarray:

    token_ids = tokenizer.encode(text, add_special_tokens=False)

    if len(token_ids) <= chunk_size:
        return get_bert_embedding_single(text)

    stride = chunk_size - overlap
    chunks = []
    for start in range(0, len(token_ids), stride):
        end       = start + chunk_size
        chunk_ids = token_ids[start:end]
        chunk_ids = [tokenizer.cls_token_id] + chunk_ids + [tokenizer.sep_token_id]
        chunks.append(chunk_ids)
        if end >= len(token_ids):
            break

    chunk_embeddings = []
    for chunk_ids in chunks:
        input_ids      = torch.tensor([chunk_ids]).to(device)
        attention_mask = torch.ones_like(input_ids).to(device)

        with torch.no_grad():
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)

        token_embeddings = outputs.last_hidden_state[0][1:-1]
        chunk_emb        = token_embeddings.mean(dim=0).cpu().numpy()
        chunk_embeddings.append(chunk_emb)

    return np.mean(chunk_embeddings, axis=0)


def get_embeddings_batch(texts: list, batch_size: int = 4) -> np.ndarray:
    embeddings = []
    for i, text in enumerate(texts):
        emb = get_bert_embedding(text)
        embeddings.append(emb)
        print(f'  Embedded {i+1}/{len(texts)}', end='\r')
    print()
    return np.array(embeddings)


sample_ids = tokenizer.encode(X_train[0], add_special_tokens=False)
stride     = CHUNK_SIZE - CHUNK_OVERLAP
n_chunks   = len(range(0, len(sample_ids), stride))
print(f'Sampel dokumen: {len(sample_ids)} token')
print(f'Jumlah chunk  : {n_chunks}')
print(f'Embedding shape: {get_bert_embedding(X_train[0]).shape}')
print('Fungsi embedding siap.')

Sampel dokumen: 39129 token
Jumlah chunk  : 112


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Embedding shape: (768,)
Fungsi embedding siap.


In [13]:
EMBED_PATH = os.path.join(MODEL_DIR, 'train_embeddings.npy')

if os.path.exists(EMBED_PATH):
    print('Cache ditemukan. Membangun ulang dengan chunking...')
    os.remove(EMBED_PATH)

print('Membangun embeddings...')
train_embeddings = get_embeddings_batch(X_train)
np.save(EMBED_PATH, train_embeddings)
print(f'Embeddings disimpan: {EMBED_PATH}')
print(f'Shape embeddings: {train_embeddings.shape}')

Membangun embeddings...
  Embedded 28/28
Embeddings disimpan: C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\models\train_embeddings.npy
Shape embeddings: (28, 768)


In [14]:
def retrieve_bert(query: str, k: int = 5) -> List[Tuple[str, float]]:
    query_clean     = query.lower().strip()
    query_embedding = get_bert_embedding(query_clean).reshape(1, -1)
    similarities    = cosine_similarity(query_embedding, train_embeddings).flatten()
    top_k_idx       = similarities.argsort()[::-1][:k]
    return [
        (train_df.iloc[idx]['case_id'], float(similarities[idx]))
        for idx in top_k_idx
    ]

print('Test IndoBERT retrieve (chunked):')
for cid, score in retrieve_bert(X_test[0], k=5):
    print(f'  {cid}: {score:.4f}')

Test IndoBERT retrieve (chunked):
  case_029: 0.9639
  case_030: 0.9627
  case_028: 0.9457
  case_013: 0.9455
  case_007: 0.9442


## 6. Fungsi Retrieve Unified

In [15]:
def retrieve(query: str, k: int = 5, method: str = 'bert') -> List[Tuple[str, float]]:
    if method == 'tfidf':
        return retrieve_tfidf(query, k)
    elif method == 'bert':
        return retrieve_bert(query, k)
    else:
        raise ValueError(f'Method tidak dikenal: {method}. Pilih tfidf atau bert.')


print('=== Demo Retrieve ===')
query_text = X_test[0]
query_case = test_df.iloc[0]['case_id']
print(f'Query dari: {query_case} (label: {test_df.iloc[0]["label_svm"]})')
print()

print('--- TF-IDF ---')
for cid, score in retrieve(query_text, k=5, method='tfidf'):
    label = train_df[train_df['case_id']==cid]['label_svm'].values
    print(f'  {cid} | score: {score:.4f} | label: {label[0] if len(label) else "-"}')

print()
print('--- IndoBERT (chunked) ---')
for cid, score in retrieve(query_text, k=5, method='bert'):
    label = train_df[train_df['case_id']==cid]['label_svm'].values
    print(f'  {cid} | score: {score:.4f} | label: {label[0] if len(label) else "-"}')

print()
print('--- SVM Prediksi Label ---')
pred_label = svm_pipeline.predict([query_text])[0]
pred_proba = svm_pipeline.predict_proba([query_text])[0]
print(f'  Prediksi: {pred_label} | Confidence: {max(pred_proba):.4f}')

=== Demo Retrieve ===
Query dari: case_009 (label: pasal 263 ayat (1))

--- TF-IDF ---
  case_014 | score: 0.4861 | label: pasal 263 ayat (2)
  case_029 | score: 0.4716 | label: pasal 263 ayat (1)
  case_030 | score: 0.4627 | label: pasal 263 ayat (1)
  case_011 | score: 0.4566 | label: pasal 263 ayat (1)
  case_019 | score: 0.4552 | label: pasal 263 ayat (1)

--- IndoBERT (chunked) ---
  case_029 | score: 0.9639 | label: pasal 263 ayat (1)
  case_030 | score: 0.9627 | label: pasal 263 ayat (1)
  case_028 | score: 0.9457 | label: pasal 263 ayat (1)
  case_013 | score: 0.9455 | label: pasal 263 ayat (2)
  case_007 | score: 0.9442 | label: pasal 263 ayat (1)

--- SVM Prediksi Label ---
  Prediksi: pasal 263 ayat (1) | Confidence: 0.5889


## 7. Buat Ground-Truth queries.json


In [16]:
queries = []

for _, row in test_df.iterrows():
    query_label = row['pasal_terbukti']

    relevant_cases = train_df[
        train_df['pasal_terbukti'] == query_label
    ]['case_id'].tolist()

    queries.append({
        'query_id'        : row['case_id'],
        'query_text'      : row['text_full'][:300],
        'no_perkara'      : row.get('no_perkara', ''),
        'label'           : query_label,
        'ground_truth_ids': relevant_cases
    })

QUERIES_PATH = os.path.join(EVAL_DIR, 'queries.json')
with open(QUERIES_PATH, 'w', encoding='utf-8') as f:
    json.dump(queries, f, ensure_ascii=False, indent=2)

print(f'queries.json disimpan: {QUERIES_PATH}')
print(f'Total queries: {len(queries)}')
print()
sample = queries[0].copy()
sample['query_text'] = sample['query_text'][:100] + '...'
print(json.dumps(sample, ensure_ascii=False, indent=2))

queries.json disimpan: C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\eval\queries.json
Total queries: 8

{
  "query_id": "case_009",
  "query_text": "putusan\nnomor 1298/pid.b/2019/pn jkt.utr.\n demi keadilan berdasarkan ketuhanan yang maha esa;\n penga...",
  "no_perkara": "1298/pid.b/2019/pn jkt.utr.",
  "label": "pasal 263 ayat (1)",
  "ground_truth_ids": [
    "case_030",
    "case_007",
    "case_036",
    "case_027",
    "case_028",
    "case_029",
    "case_005",
    "case_023",
    "case_002",
    "case_011",
    "case_003",
    "case_006",
    "case_004",
    "case_026",
    "case_019"
  ]
}


## 8. Pengujian Awal — Evaluasi Retrieval

In [17]:
def precision_at_k(retrieved, ground_truth, k):
    retrieved_ids = [cid for cid, _ in retrieved[:k]]
    hits = sum(1 for cid in retrieved_ids if cid in ground_truth)
    return hits / k

def recall_at_k(retrieved, ground_truth, k):
    if not ground_truth:
        return 0.0
    retrieved_ids = [cid for cid, _ in retrieved[:k]]
    hits = sum(1 for cid in retrieved_ids if cid in ground_truth)
    return hits / len(ground_truth)

results_summary = []

for q in queries:
    query_text   = test_df[test_df['case_id'] == q['query_id']]['text_full'].values[0]
    ground_truth = q['ground_truth_ids']

    tfidf_results = retrieve(query_text, k=5, method='tfidf')
    bert_results  = retrieve(query_text, k=5, method='bert')
    svm_pred      = svm_pipeline.predict([query_text])[0]

    results_summary.append({
        'query_id'   : q['query_id'],
        'true_label' : q['label'],
        'svm_pred'   : svm_pred,
        'svm_correct': int(svm_pred == q['label']),
        'p@5_tfidf'  : precision_at_k(tfidf_results, ground_truth, k=5),
        'r@5_tfidf'  : recall_at_k(tfidf_results, ground_truth, k=5),
        'p@5_bert'   : precision_at_k(bert_results, ground_truth, k=5),
        'r@5_bert'   : recall_at_k(bert_results, ground_truth, k=5),
        'top5_tfidf' : [cid for cid, _ in tfidf_results],
        'top5_bert'  : [cid for cid, _ in bert_results],
    })

results_df = pd.DataFrame(results_summary)

print('=== Hasil Pengujian Awal ===')
print(results_df[['query_id', 'true_label', 'svm_pred', 'svm_correct',
                   'p@5_tfidf', 'p@5_bert']].to_string(index=False))
print()
print(f'SVM Accuracy           : {results_df["svm_correct"].mean():.4f}')
print(f'Mean Precision@5 TF-IDF: {results_df["p@5_tfidf"].mean():.4f}')
print(f'Mean Precision@5 BERT  : {results_df["p@5_bert"].mean():.4f}')
print(f'Mean Recall@5 TF-IDF   : {results_df["r@5_tfidf"].mean():.4f}')
print(f'Mean Recall@5 BERT     : {results_df["r@5_bert"].mean():.4f}')

=== Hasil Pengujian Awal ===
query_id         true_label           svm_pred  svm_correct  p@5_tfidf  p@5_bert
case_009 pasal 263 ayat (1) pasal 263 ayat (1)            1        0.8       0.8
case_018 pasal 266 ayat (1)          pasal 266            0        0.4       0.6
case_034              bebas              bebas            1        0.4       0.2
case_025              bebas              bebas            1        0.4       0.4
case_010 pasal 263 ayat (1) pasal 263 ayat (1)            1        0.8       1.0
case_020 pasal 263 ayat (1) pasal 263 ayat (1)            1        1.0       1.0
case_032 pasal 263 ayat (1) pasal 263 ayat (1)            1        0.8       1.0
case_015 pasal 263 ayat (2) pasal 263 ayat (1)            0        0.2       0.2

SVM Accuracy           : 0.7500
Mean Precision@5 TF-IDF: 0.6000
Mean Precision@5 BERT  : 0.6500
Mean Recall@5 TF-IDF   : 0.3604
Mean Recall@5 BERT     : 0.3771


In [18]:
RESULTS_PATH = os.path.join(EVAL_DIR, 'retrieval_initial_test.csv')
results_df.drop(columns=['top5_tfidf', 'top5_bert']).to_csv(RESULTS_PATH, index=False)
print(f'Hasil disimpan: {RESULTS_PATH}')

Hasil disimpan: C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\eval\retrieval_initial_test.csv


## 9. Simpan Semua Model

In [19]:
TFIDF_RETRIEVER_PATH = os.path.join(MODEL_DIR, 'tfidf_retriever.pkl')
with open(TFIDF_RETRIEVER_PATH, 'wb') as f:
    pickle.dump(tfidf_retriever, f)
print(f'TF-IDF retriever disimpan: {TFIDF_RETRIEVER_PATH}')

TFIDF_MATRIX_PATH = os.path.join(MODEL_DIR, 'train_tfidf_matrix.pkl')
with open(TFIDF_MATRIX_PATH, 'wb') as f:
    pickle.dump(train_tfidf, f)
print(f'TF-IDF matrix disimpan   : {TFIDF_MATRIX_PATH}')

SVM_PATH = os.path.join(MODEL_DIR, 'svm_pipeline.pkl')
with open(SVM_PATH, 'wb') as f:
    pickle.dump(svm_pipeline, f)
print(f'SVM pipeline disimpan    : {SVM_PATH}')

BERT_DIR = os.path.join(MODEL_DIR, 'indobert')
tokenizer.save_pretrained(BERT_DIR)
bert_model.save_pretrained(BERT_DIR)
print(f'IndoBERT disimpan        : {BERT_DIR}/')

np.save(EMBED_PATH, train_embeddings)
print(f'Train embeddings disimpan: {EMBED_PATH}')

TRAIN_DF_PATH = os.path.join(MODEL_DIR, 'train_df.pkl')
with open(TRAIN_DF_PATH, 'wb') as f:
    pickle.dump(train_df, f)
print(f'Train dataframe disimpan : {TRAIN_DF_PATH}')

TEST_DF_PATH = os.path.join(MODEL_DIR, 'test_df.pkl')
with open(TEST_DF_PATH, 'wb') as f:
    pickle.dump(test_df, f)
print(f'Test dataframe disimpan  : {TEST_DF_PATH}')

print()
print('=== Semua model berhasil disimpan. ===')

TF-IDF retriever disimpan: C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\models\tfidf_retriever.pkl
TF-IDF matrix disimpan   : C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\models\train_tfidf_matrix.pkl
SVM pipeline disimpan    : C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\models\svm_pipeline.pkl


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

IndoBERT disimpan        : C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\models\indobert/
Train embeddings disimpan: C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\models\train_embeddings.npy
Train dataframe disimpan : C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\models\train_df.pkl
Test dataframe disimpan  : C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\models\test_df.pkl

=== Semua model berhasil disimpan. ===
